# 1 环境依赖

这里使用与cellpose一致的环境

Follow the installation steps on github https://github.com/MouseLand/cellpose

Here shows the installation steps for Linux server

1. creat a new environment with `conda create --name cellpose python=3.10`.
2. Activate this new environment with `conda activate cellpose`.
3. To install cellpose without the GUI, run `python -m pip install cellpose`. 


# 2 数据目录结构

沿用Code1中得到的数据目录，在每个slides的独立文件夹下新建`segmentation/cellpose_input`文件夹用来保存归一化合并后的输入图像

```
Experiment Folder
|
|---Slides1.ome.tif
|---Slides2.ome.tif
|---Slides3.ome.tif
...
|---Slides1
|   |---Tiles
|       |---FOV0.tif
|       |---FOV1.tif
|       ...
|   |---image_data
|       |---FOV0
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|       |---FOV1
|       |   |---channel1.tif
|       |   |---channel2.tif
|       |   |---channel3.tif
|       |   ...
|   |---segementation
|       |---cellpose_input
|           |---FOV0.tif
|           |---FOV1.tif
...
```


# 3 处理逻辑

## Signal Prepare

需要用户指定核信号的marker名称和膜信号的marker名称，marker名称需要和code1中Split Channel定义的`Channel Name`保持一致，如果不存在需要报错。

> ### Parameter
> nuclear_markers: list
> 
> membrane_markers: list
>
> ### Input
> ./Experiment Folder/Slides Name/FOVx/channelx.tif
>
> ### Output
> ./Experiment Folder/Slides Name/segmentation/cellpose_input/FOVx.tif

# 4 原始代码

/home/jqyang/cellpose/demo/demo.ipynb

/home/jqyang/cellpose/demo/demo_batch.ipynb

In [1]:
import os
import numpy as np
import tifffile
from glob import glob

def normalize_channel(image_data, lower_percentile=1.0, upper_percentile=99.0):
    img_flat = image_data.flatten()
    nonzero_pixels = img_flat[img_flat > 0]
    
    if len(nonzero_pixels) == 0:
        return np.zeros_like(image_data, dtype=np.float32)
        
    p_low = np.percentile(nonzero_pixels, lower_percentile)
    p_high = np.percentile(nonzero_pixels, upper_percentile)
    
    if p_high - p_low == 0:
        return np.zeros_like(image_data, dtype=np.float32)
        
    img_norm = (image_data.astype(np.float32) - p_low) / (p_high - p_low)
    return np.clip(img_norm, 0.0, 1.0)

def prepare_cellpose_inputs(base_dir, nuc_markers, mem_markers):
    image_dir = os.path.join(base_dir, "image_data")
    output_dir = os.path.join(base_dir, "segmentation", "cellpose_input")
    os.makedirs(output_dir, exist_ok=True)

    # 获取所有 fov 文件夹
    fov_folders = [f for f in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, f))]
    
    for fov in fov_folders:
        print(f"正在处理: {fov}")
        fov_path = os.path.join(image_dir, fov)
        
        # 尝试读取任意一张图获取 H 和 W
        sample_img_path = glob(os.path.join(fov_path, "*.tif"))[0]
        sample_img = tifffile.imread(sample_img_path)
        h, w = sample_img.shape
        
        out_img = np.zeros((2, h, w), dtype=np.float32)

        # 1. 处理核通道 (Channel 0)
        for marker in nuc_markers:
            img_path = os.path.join(fov_path, f"{marker}.tif")
            if os.path.exists(img_path):
                img_data = tifffile.imread(img_path)
                out_img[0] += normalize_channel(img_data)
        out_img[0] = np.clip(out_img[0], 0.0, 1.0)
                
        # 2. 处理膜/质通道 (Channel 1)
        for marker in mem_markers:
            img_path = os.path.join(fov_path, f"{marker}.tif")
            if os.path.exists(img_path):
                img_data = tifffile.imread(img_path)
                out_img[1] += normalize_channel(img_data)
        out_img[1] = np.clip(out_img[1], 0.0, 1.0)
        
        # 保存为 16-bit 双通道 TIFF [2, H, W]
        out_img_16bit = (out_img * 65535).astype(np.uint16)
        save_path = os.path.join(output_dir, f"{fov}.tiff")
        tifffile.imwrite(save_path, out_img_16bit, imagej=True)

# ===== 运行配置 =====
base_directory = "/home/jqyang/cellpose/demo/B319D_rmbg"
# 根据您的目录，DAPI是唯一的核染料，EOMES是核内转录因子(也可加入核通道，或者只用DAPI)
nuclear_markers = ["DAPI", "EOMES"] 
# 其他表面抗原作为细胞膜/质信号
membrane_markers = ["CD3", "CD4", "CD8a", "TCRbeta"] 

prepare_cellpose_inputs(base_directory, nuclear_markers, membrane_markers)

正在处理: fov37
正在处理: fov43
正在处理: fov63
正在处理: fov10
正在处理: fov31
正在处理: fov35
正在处理: fov60
正在处理: fov56
正在处理: fov62
正在处理: fov49
正在处理: fov1
正在处理: fov50
正在处理: fov3
正在处理: fov16
正在处理: fov47
正在处理: fov30
正在处理: fov17
正在处理: fov11
正在处理: fov24
正在处理: fov18
正在处理: fov33
正在处理: fov5
正在处理: fov38
正在处理: fov54
正在处理: fov61
正在处理: fov6
正在处理: fov45
正在处理: fov12
正在处理: fov8
正在处理: fov23
正在处理: fov44
正在处理: fov57
正在处理: fov51
正在处理: fov27
正在处理: fov41
正在处理: fov15
正在处理: fov4
正在处理: fov59
正在处理: fov39
正在处理: fov9
正在处理: fov55
正在处理: fov22
正在处理: fov52
正在处理: fov14
正在处理: fov0
正在处理: fov25
正在处理: fov42
正在处理: fov13
正在处理: fov48
正在处理: fov58
正在处理: fov7
正在处理: fov53
正在处理: fov40
正在处理: fov29
正在处理: fov36
正在处理: fov21
正在处理: fov34
正在处理: fov26
正在处理: fov20
正在处理: fov2
正在处理: fov46
正在处理: fov19
正在处理: fov28
正在处理: fov32
